In [0]:
import os 
import sys

project_path = os.path.join(os.getcwd(),'..','..')


sys.path.append(project_path)

from utils.Transformation import *

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#### **AUTOLOADER* ####


## **DIM USER**

In [0]:
# 1. Read Stream (with a dedicated schema tracking directory)
df_user = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://silver@anujspotifyazureproject.dfs.core.windows.net/DimUser/checkpoint")
    .option("schemaEvolutionMode", "addNewColumns")
    .load("abfss://bronze@anujspotifyazureproject.dfs.core.windows.net/DimUser")
)

# 2. Apply Transformations
df_user_obj = reusable()
df_user_cleaned = df_user_obj.dropColumns(df_user, ['_rescued_data'])
df_user_cleaned = df_user_cleaned.dropDuplicates(['user_id'])

# 3. Write Stream to Unity Catalog Table
user_query = (df_user_cleaned.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@anujspotifyazureproject.dfs.core.windows.net/DimUser/checkpoint")
    .trigger(availableNow=True) # Must be BEFORE .toTable()
    .option("path","abfss://silver@anujspotifyazureproject.dfs.core.windows.net/DimUser/data")
    .toTable("spotify_cata.silver.DimUser") # .toTable starts the stream (replaces .start)
)

## **DimArtist**


In [0]:
# 1. Read Stream (with a dedicated schema tracking directory)
df_artist = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://silver@anujspotifyazureproject.dfs.core.windows.net/DimArtist/checkpoint")
    .option("schemaEvolutionMode", "addNewColumns")
    .load("abfss://bronze@anujspotifyazureproject.dfs.core.windows.net/DimArtist")
)

# 2. Apply Transformations
df_artist_obj = reusable()
df_artist_cleaned = df_artist_obj.dropColumns(df_artist, ['_rescued_data'])
df_artist_cleaned = df_artist_cleaned.dropDuplicates(['artist_id'])

# 3. Write Stream to Unity Catalog Table
artist_query = (df_artist_cleaned.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@anujspotifyazureproject.dfs.core.windows.net/DimArtist/checkpoint")
    .trigger(availableNow=True) # Must be BEFORE .toTable()
    .option("path","abfss://silver@anujspotifyazureproject.dfs.core.windows.net/DimArtist/data")
    .toTable("spotify_cata.silver.DimArtist") # .toTable starts the stream (replaces .start)
)

##  **DimTrack**

## **Read**

In [0]:
# 1. Read Stream (with a dedicated schema tracking directory)
df_track = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://silver@anujspotifyazureproject.dfs.core.windows.net/DimTrack/checkpoint")
    .option("schemaEvolutionMode", "addNewColumns")
    .load("abfss://bronze@anujspotifyazureproject.dfs.core.windows.net/DimTrack")
)

## **Transformation**

In [0]:
# 2. Apply Transformations
df_track = df_track.withColumn("durationFlag",when(col('duration_sec')<150, "low")\
                                            .when(col('duration_sec')<300, "medium")\
                                            .otherwise("high"))

df_track = df_track.withColumn("track_name",regexp_replace(col('track_name'),'-',' '))

df_track = reusable().dropColumns(df_track, ['_rescued_data'])

## **Write**

In [0]:
# 3. Write Stream to Unity Catalog Table
track_query = (df_track.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@anujspotifyazureproject.dfs.core.windows.net/DimTrack/checkpoint")
    .trigger(availableNow=True) # Must be BEFORE .toTable()
    .option("path","abfss://silver@anujspotifyazureproject.dfs.core.windows.net/DimTrack/data")
    .toTable("spotify_cata.silver.DimTrack") # .toTable starts the stream (replaces .start)
)

## **DimDate**

In [0]:
# 1. Read Stream (with a dedicated schema tracking directory)
df_date = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://silver@anujspotifyazureproject.dfs.core.windows.net/DimDate/checkpoint")
    .option("schemaEvolutionMode", "addNewColumns")
    .load("abfss://bronze@anujspotifyazureproject.dfs.core.windows.net/DimDate")
)

# Transformation

df_date = reusable().dropColumns(df_date, ['_rescued_data'])

# Write SparkStreaming

# 3. Write Stream to Unity Catalog Table
date_query = (df_date.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@anujspotifyazureproject.dfs.core.windows.net/DimDate/checkpoint")
    .trigger(availableNow=True) # Must be BEFORE .toTable()
    .option("path","abfss://silver@anujspotifyazureproject.dfs.core.windows.net/DimDate/data")
    .toTable("spotify_cata.silver.DimDate") # .toTable starts the stream (replaces .start)
)


## **FactStream**

In [0]:
# 1. Read Stream (with a dedicated schema tracking directory)
df_fact = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://silver@anujspotifyazureproject.dfs.core.windows.net/FactStream/checkpoint")
    .option("schemaEvolutionMode", "addNewColumns")
    .load("abfss://bronze@anujspotifyazureproject.dfs.core.windows.net/FactStream")
)

## **Transformation** + **Write_In_Target**

In [0]:
#Transformartion

df_fact = reusable().dropColumns(df_fact, ['_rescued_data'])


# 3. Write Stream to Unity Catalog Table
fact_query = (df_fact.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@anujspotifyazureproject.dfs.core.windows.net/FactStream/checkpoint")
    .trigger(availableNow=True) # Must be BEFORE .toTable()
    .option("path","abfss://silver@anujspotifyazureproject.dfs.core.windows.net/FactStream/data")
    .toTable("spotify_cata.silver.FactStream") # .toTable starts the stream (replaces .start)
)



## **EXTRA**